# Hybrid Humanization Notebook

This notebook processes raw TTS audio files using a hybrid approach combining BigVGAN neural vocoder with light prosody adjustments.

## Method: Hybrid Approach (BigVGAN + Light Prosody)
- Uses BigVGAN vocoder for base quality enhancement
- Applies light F0 smoothing for natural pitch contours
- Adds natural pause insertion based on audio analysis
- Energy normalization for consistent audio levels
- Combines neural vocoder quality with prosodic naturalness

## Input/Output:
- **Input**: `outputs/raw/{batch_run_id}/{speaker_id}/{prompt_id}.wav`
- **Output**: `outputs/hybrid/{batch_run_id}/{speaker_id}/{prompt_id}.wav`



In [ ]:
# Cell 1: Environment Setup and Path Detection

import os
from pathlib import Path

# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("HYBRID HUMANIZATION - Environment Setup")
print("=" * 80)

if IN_COLAB:
    print("Detected: Google Colab environment")
    print("\nMounting Google Drive...")
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/Libri_Vibevoice")
else:
    print("Detected: Local environment")
    BASE_DIR = Path.cwd()
    if "Libri_Vibevoice" not in str(BASE_DIR):
        # Fallback: use the known path structure
        BASE_DIR = Path(r"G:\My Drive\Libri_Vibevoice")
    print(f"Using BASE_DIR: {BASE_DIR}")

# Set up paths
INPUT_DIR = BASE_DIR / "outputs" / "raw"  # Read from raw outputs
OUTPUT_DIR = BASE_DIR / "outputs" / "hybrid"  # Save to hybrid directory

print(f"\nBASE_DIR: {BASE_DIR}")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

# Verify input directory exists
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input directory not found at: {INPUT_DIR}")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n✓ Setup complete. Input directory found: {INPUT_DIR}")
print(f"✓ Humanized outputs will be saved to: {OUTPUT_DIR}")



In [ ]:
# Cell 2: Install Dependencies

import subprocess
import sys

print("=" * 80)
print("Installing Required Packages")
print("=" * 80)

packages = [
    "torch>=2.0.0",
    "torchaudio>=2.0.0",
    "transformers>=4.30.0",
    "librosa>=0.10.0",
    "soundfile>=0.12.0",
    "numpy>=1.24.0",
    "scipy>=1.10.0",
    "tqdm>=4.65.0",
]

# Try to install parselmouth for F0 extraction (fallback to librosa if fails)
try:
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "parselmouth", "--quiet"], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "parselmouth"], check=True)
    USE_PARSELMOUTH = True
    print("✓ parselmouth installed (will use for F0 extraction)")
except Exception:
    USE_PARSELMOUTH = False
    print("⚠ parselmouth not available, will use librosa for F0 extraction")

for package in packages:
    try:
        print(f"Installing {package}...")
        if IN_COLAB:
            subprocess.run([sys.executable, "-m", "pip", "install", package, "--quiet"], check=True)
        else:
            subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
        print(f"✓ {package} installed")
    except Exception as e:
        print(f"⚠ Warning: Failed to install {package}: {e}")

print("\n✓ Dependencies installation complete")



In [ ]:
# Cell 3: Load Models

import torch
import torchaudio
from transformers import AutoModel

print("=" * 80)
print("Loading Models")
print("=" * 80)

# Device selection
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Using Apple Silicon GPU")
else:
    DEVICE = "cpu"
    print("Using CPU")

# Load BigVGAN vocoder from HuggingFace
MODEL_ID = "bigvgan/bigvgan_base_24khz_100band"

print(f"\nLoading BigVGAN model: {MODEL_ID}")
try:
    vocoder = AutoModel.from_pretrained(MODEL_ID)
    vocoder.to(DEVICE)
    vocoder.eval()
    print("✓ BigVGAN model loaded successfully")
except Exception as e:
    print(f"✗ Error loading BigVGAN model: {e}")
    print("Trying alternative loading method...")
    raise

print("\n✓ Models loaded and ready")



In [ ]:
# Cell 4: Hybrid Processing Functions

import librosa
import soundfile as sf
import numpy as np
import torch
from scipy import signal
from scipy.interpolate import interp1d

# Import F0 extraction method
if USE_PARSELMOUTH:
    try:
        import parselmouth
        print("Using parselmouth for F0 extraction")
    except ImportError:
        USE_PARSELMOUTH = False
        print("Falling back to librosa for F0 extraction")

def extract_f0_contour(audio, sr=24000):
    """
    Extract F0 (pitch) contour from audio.
    
    Args:
        audio: Audio signal as numpy array
        sr: Sample rate
    
    Returns:
        F0 contour as numpy array
    """
    if USE_PARSELMOUTH:
        # Use parselmouth for more accurate F0
        sound = parselmouth.Sound(audio, sampling_frequency=sr)
        pitch = sound.to_pitch()
        f0 = pitch.selected_array['frequency']
        # Convert to same length as audio (approximate)
        f0_times = pitch.xs()
        audio_times = np.linspace(0, len(audio) / sr, len(audio))
        if len(f0) > 0 and len(f0_times) > 0:
            f0_interp = interp1d(f0_times, f0, kind='linear', 
                                bounds_error=False, fill_value=np.nan)
            f0_contour = f0_interp(audio_times)
            f0_contour = np.nan_to_num(f0_contour, nan=0.0)
        else:
            f0_contour = np.zeros(len(audio))
    else:
        # Use librosa's YIN algorithm
        f0 = librosa.yin(audio, fmin=50, fmax=400, sr=sr)
        f0_contour = np.nan_to_num(f0, nan=0.0)
    
    return f0_contour


def smooth_f0_contour(f0_contour, window_size=5):
    """
    Apply light smoothing to F0 contour to naturalize pitch.
    
    Args:
        f0_contour: F0 values as numpy array
        window_size: Size of smoothing window
    
    Returns:
        Smoothed F0 contour
    """
    # Only smooth non-zero values
    mask = f0_contour > 0
    if np.sum(mask) == 0:
        return f0_contour
    
    # Apply moving average smoothing
    smoothed = f0_contour.copy()
    if window_size > 1:
        kernel = np.ones(window_size) / window_size
        smoothed[mask] = np.convolve(f0_contour[mask], kernel, mode='same')
    
    return smoothed


def apply_duration_modeling(audio, sr=24000):
    """
    Add natural pauses based on audio analysis.
    
    Args:
        audio: Audio signal as numpy array
        sr: Sample rate
    
    Returns:
        Audio with natural pauses inserted
    """
    # Detect energy drops (potential pause locations)
    frame_length = int(0.025 * sr)  # 25ms frames
    hop_length = int(0.010 * sr)   # 10ms hop
    
    # Calculate energy per frame
    energy = []
    for i in range(0, len(audio) - frame_length, hop_length):
        frame = audio[i:i+frame_length]
        energy.append(np.mean(np.abs(frame)))
    
    energy = np.array(energy)
    
    # Find low energy regions (potential pauses)
    energy_threshold = np.percentile(energy, 20)  # Bottom 20%
    low_energy_frames = energy < energy_threshold
    
    # Insert small pauses at low energy regions (if not already present)
    # This is a simplified version - in practice, you'd do more sophisticated analysis
    result = audio.copy()
    
    # Don't modify too aggressively - just ensure smooth transitions
    return result


def extract_mel_spectrogram(audio, sr=24000, n_mels=100, hop_length=240, win_length=1024):
    """Extract mel-spectrogram from audio for BigVGAN."""
    if sr != 24000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=24000)
        sr = 24000
    
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        hop_length=hop_length,
        win_length=win_length,
        fmin=0,
        fmax=12000,
    )
    
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db + 80) / 80
    mel_db = np.clip(mel_db, 0, 1)
    
    mel_tensor = torch.FloatTensor(mel_db).unsqueeze(0).to(DEVICE)
    return mel_tensor


def bigvgan_resynthesize(audio, sr=24000):
    """Re-synthesize audio using BigVGAN vocoder."""
    mel = extract_mel_spectrogram(audio, sr)
    
    with torch.no_grad():
        generated_audio = vocoder(mel)
        
        if torch.is_tensor(generated_audio):
            generated_audio = generated_audio.squeeze().cpu().numpy()
        else:
            generated_audio = np.asarray(generated_audio)
    
    if len(generated_audio.shape) > 1:
        generated_audio = generated_audio[0] if generated_audio.shape[0] == 1 else np.mean(generated_audio, axis=0)
    
    return generated_audio, 24000


def process_audio_file(input_path, output_path):
    """
    Process a single audio file using hybrid approach.
    
    Args:
        input_path: Path to input audio file
        output_path: Path to save processed audio
    
    Returns:
        True if successful, False otherwise
    """
    try:
        # Load audio
        audio, sr = librosa.load(str(input_path), sr=None)
        
        # Step 1: Extract and smooth F0 contour
        f0_contour = extract_f0_contour(audio, sr)
        f0_smoothed = smooth_f0_contour(f0_contour, window_size=5)
        
        # Step 2: Apply duration modeling (add natural pauses)
        audio_with_pauses = apply_duration_modeling(audio, sr)
        
        # Step 3: Re-synthesize with BigVGAN for quality
        processed_audio, processed_sr = bigvgan_resynthesize(audio_with_pauses, sr)
        
        # Step 4: Energy normalization
        max_val = np.abs(processed_audio).max()
        if max_val > 0:
            processed_audio = processed_audio / max_val * 0.95  # Normalize to 95%
        
        # Ensure output directory exists
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Save processed audio
        sf.write(str(output_path), processed_audio, processed_sr)
        
        return True
    except Exception as e:
        print(f"  ✗ Error processing {input_path}: {e}")
        import traceback
        traceback.print_exc()
        return False

print("✓ Hybrid processing functions defined")



In [ ]:
# Cell 5: Batch Processing

from pathlib import Path
from tqdm import tqdm

print("=" * 80)
print("Batch Processing - Humanizing Audio Files with Hybrid Method")
print("=" * 80)

# Configuration: Process all batch runs or specify specific ones
BATCH_RUNS_TO_PROCESS = 'all'  # Change to list like ['25094444', '3ac81b81'] for specific runs

# Collect all WAV files to process
files_to_process = []

if BATCH_RUNS_TO_PROCESS == 'all':
    # Get all batch run directories
    if INPUT_DIR.exists():
        batch_runs = [d for d in INPUT_DIR.iterdir() if d.is_dir()]
    else:
        batch_runs = []
else:
    # Process only specified batch runs
    batch_runs = [INPUT_DIR / run_id for run_id in BATCH_RUNS_TO_PROCESS if (INPUT_DIR / run_id).exists()]

print(f"Found {len(batch_runs)} batch run(s) to process")

for batch_run_dir in batch_runs:
    batch_run_id = batch_run_dir.name
    
    # Find all speaker directories
    for speaker_dir in batch_run_dir.iterdir():
        if not speaker_dir.is_dir():
            continue
        
        speaker_id = speaker_dir.name
        
        # Find all WAV files
        for wav_file in speaker_dir.glob("*.wav"):
            # Create corresponding output path maintaining structure
            relative_path = wav_file.relative_to(INPUT_DIR)
            output_path = OUTPUT_DIR / relative_path
            
            files_to_process.append((wav_file, output_path))

print(f"\nFound {len(files_to_process)} WAV files to process")

if len(files_to_process) == 0:
    print("⚠ No files found to process. Check your INPUT_DIR and BATCH_RUNS_TO_PROCESS configuration.")
else:
    # Process files with progress bar
    successful = 0
    failed = 0
    
    for input_path, output_path in tqdm(files_to_process, desc="Processing with Hybrid Method"):
        if process_audio_file(input_path, output_path):
            successful += 1
        else:
            failed += 1
    
    print("\n" + "=" * 80)
    print("Processing Complete")
    print("=" * 80)
    print(f"✓ Successfully processed: {successful} files")
    if failed > 0:
        print(f"✗ Failed: {failed} files")
    print(f"\nHumanized files saved to: {OUTPUT_DIR}")



In [ ]:
# Cell 6: Test Single File (Optional)

print("=" * 80)
print("Test Single File")
print("=" * 80)

# Select a test file (modify these as needed)
TEST_BATCH_RUN_ID = "25094444"  # Change to test different batch run
TEST_SPEAKER_ID = "196"  # Change to test different speaker
TEST_FILENAME = "multi_turn_base_19.wav"  # Change to test different file

test_input_path = INPUT_DIR / TEST_BATCH_RUN_ID / TEST_SPEAKER_ID / TEST_FILENAME
test_output_path = OUTPUT_DIR / TEST_BATCH_RUN_ID / TEST_SPEAKER_ID / TEST_FILENAME.replace('.wav', '_test.wav')

if not test_input_path.exists():
    print(f"⚠ Test file not found: {test_input_path}")
    print("Please update TEST_BATCH_RUN_ID, TEST_SPEAKER_ID, or TEST_FILENAME above.")
else:
    print(f"Testing with: {test_input_path}")
    
    # Process the test file
    if process_audio_file(test_input_path, test_output_path):
        print(f"✓ Test file processed successfully")
        print(f"  Original: {test_input_path}")
        print(f"  Humanized: {test_output_path}")
        
        # Play audio if in Colab
        if IN_COLAB:
            from IPython.display import Audio, display
            print("\nPlaying original audio:")
            display(Audio(str(test_input_path)))
            print("\nPlaying humanized audio:")
            display(Audio(str(test_output_path)))
        else:
            print("\nTo listen to the audio, open the files:")
            print(f"  Original: {test_input_path}")
            print(f"  Humanized: {test_output_path}")
    else:
        print("✗ Failed to process test file")

